In [ ]:
# Import python packages
import streamlit as st
import pandas as pd

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
#PREPROCESS SCRIPT

import re
import pandas as pd
from snowflake.snowpark.functions import upper, col

def clean_artist_name(name):
    """
    Cleans artist names by:
    1. Replacing special characters with spaces
    2. Removing articles (A, AN, THE) and common words like 'BAND', 'ORCHESTRA'
    3. Removing all non-alphanumeric characters
    4. Normalizing spaces
    5. Removing parenthetical content
    """
    if name is None or pd.isna(name):
        return ""
    
    try:
        # Convert to uppercase for consistent processing
        name = str(name).upper()
        
        # Create a copy of the name without apostrophes for cleaning
        name_no_apostrophes = re.sub(r'[\']', '', name)  # Remove apostrophes for processing
        
        # Replace special characters with spaces to preserve word boundaries
        # Added hyphen to the special characters list
        name_with_spaces = re.sub(r'["`\-_&+\]\[]', '', name_no_apostrophes)
        
        # Remove remaining non-alphanumeric characters (except spaces)
        cleaned = re.sub(r'[^a-zA-Z0-9 ]', '', name_with_spaces)
        
        # Normalize spaces (convert multiple spaces to single space)
        cleaned = re.sub(r'\s+', ' ', cleaned).strip()
        
        # If the result is just spaces, return the original name
        if cleaned == "":
            return name
        
        # Updated stop_words list with additional normalizations
        stop_words = [
            # Basic articles
            r'\bA\b', r'\bAN\b', r'\bTHE\b',
            
            # Original orchestra/band terms
            r'\bAKA\b', r'\bBAND\b', r'\bFEAT\b', r'\bFEAT\.\b', r'\bFT\b', r'\bFT\.\b',
            r'\bAND\b', r'\bORCHESTRA\b', r'\bORCHES\b', r'\bQUARTET\b', r'\bWITH\b', r'\bHIS\b', r'\bHER\b',
            r'\bCHOIR\b', r'\bTRIO\b', r'\bPHILHARMONIC ORCHESTRA\b', r'\bPHILHARMONIC ORCHES\b', 
            r'\bPHILHARMONIC\b', r'\bSYMPHONY\b', r'\bDUO\b', r'\bKARAOKE\b', r'\bKARAOKE GROUP\b', 
            r'\bBRASS BAND\b',
            
            # New normalizations from provided list
            r'\b24 BIT DIGITAL REMASTER\b', r'\b24 BIT REMASTER\b',
            r'\bACOUSTIC VERSION\b', r'\bALTERNATE MIX\b', r'\bALTERNATE TAKE\b',
            r'\bCOMPILATION\b', r'\bDEMO VERSION\b', r'\bDIGITAL REMASTER\b',
            r'\bKARAOKE\b', r'\bLIVE\b', r'\bMASTER TAKE\b',
            r'\bMONO MIX\b', r'\bMONO VERSION\b',
            r'\bN/S\b', r'\bNON IDENTIFICATED\b', r'\bNONE\b',
            r'\bNOR IDENTIFIED\b', r'\bNOT IDENTIFIED\b', r'\bNOT KNOWN\b',
            r'\bNOT SHOWN\b', r'\bNOT SPECIFIED\b', r'\bNOT STATED\b',
            r'\bNS\b', r'\bO/S/T\b', r'\bORCH\b',
            r'\bORCHESTRA\b', r'\bORIGINAL SOUND TRACK\b', r'\bORIGINAL SOUNDTRACK\b',
            r'\bOST\b', r'\bPROMO\b', r'\bPROMO VERSION\b',
            r'\bREHEARSAL\b', r'\bREMAKE\b', r'\bREMASTERED\b',
            r'\bS/T\b', r'\bSOUNDTRACK\b', r'\bST\b',
            r'\bSTEREO MIX\b', r'\bSTUDIO REHEARSAL\b', r'\bTHE UNKNOWN\b',
            r'\bTRADITIONAL\b', r'\bTRIBUTE BAND\b', r'\bUNKNOWN\b',
            r'\bUNKNOWN ARTIST\b', r'\bVAR\b', r'\bVARIOUS\b',
            r'\bVARIOUS ARTIST\b', r'\bVARIOUS ARTISTS\b',
            r'\bLIVE MUSIC FOR DISTRIBUTABLE EVENTS\b',
            r'\bRECORDED MUSIC FOR DISTRIBUTABLE EVENTS\b'
        ]
        
        for word in stop_words:
            cleaned = re.sub(word, ' ', cleaned)
        
        # Normalize spaces again after removing stop words
        cleaned = re.sub(r'\s+', ' ', cleaned).strip()
        
        # If the result is empty after removing stop words, return the previous cleaned version
        if cleaned == "":
            return name_with_spaces.strip()
        
        return cleaned
    except:
        # If any error occurs, return empty string
        return ""


def normalize_special_chars(text):
    
    if not text:
        return ""
        
    try:
        text = str(text).upper()
        
        # Remove leading brackets and braces
        text = re.sub(r'^[\[\(\{]', '', text)
        
        # Replace apostrophes, hyphens and similar characters with empty string
        text = re.sub(r'[\'"`\-]', '', text)
        
        # Replace accented characters with their basic equivalents
        chars_map = {
            'Ø': 'O', 'Ö': 'O', 'Ó': 'O', 'Ò': 'O', 'Ô': 'O', 'Õ': 'O',
            'Ä': 'A', 'Á': 'A', 'À': 'A', 'Â': 'A', 'Ã': 'A',
            'Ë': 'E', 'É': 'E', 'È': 'E', 'Ê': 'E',
            'Ü': 'U', 'Ú': 'U', 'Ù': 'U', 'Û': 'U',
            'Ï': 'I', 'Í': 'I', 'Ì': 'I', 'Î': 'I',
            'Ñ': 'N', 'Ç': 'C'
        }
        for special, basic in chars_map.items():
            text = text.replace(special, basic)
        
        # Word translations (mainly US to UK English and common abbreviations)
        # Using word boundaries to ensure we only replace whole words
        word_translations = {
            r'\bAND\b': '&',
            r'\bACKNOWLEDGMENT\b': 'ACKNOWLEDGEMENT',
            r'\bAIRPLANE\b': 'AEROPLANE',
            r'\bESTHETIC\b': 'AESTHETIC',
            r'\bAGING\b': 'AGEING',
            r'\bALUMINUM\b': 'ALUMINIUM',
            r'\bAMEBA\b': 'AMOEBA',
            r'\bANEMIA\b': 'ANAEMIA',
            r'\bANESTHESIA\b': 'ANAESTHESIA',
            r'\bANALYZE\b': 'ANALYSE',
            r'\bANALOG\b': 'ANALOGUE',
            r'\bANNEX\b': 'ANNEXE',
            r'\bANY ONE\b': 'ANYONE',
            r'\bAPOLOGIZE\b': 'APOLOGISE',
            r'\bARCHEOLOGY\b': 'ARCHAEOLOGY',
            r'\bARMOR\b': 'ARMOUR',
            r'\bASS\b': 'ARSE',
            r'\bARTIFACT\b': 'ARTEFACT',
            r'\bAUTHORIZE\b': 'AUTHORISE',
            r'\bAX\b': 'AXE',
            r'\bBEHAVIOR\b': 'BEHAVIOUR',
            r'\bBRONCO\b': 'BRONCHO',
            r'\bCESIUM\b': 'CAESIUM',
            r'\bCAMOMILE\b': 'CHAMOMILE',
            r'\bCANCELED\b': 'CANCELLED',
            r'\bCARBURETOR\b': 'CARBURETTOR',
            r'\bCATALOG\b': 'CATALOGUE',
            r'\bCENTER\b': 'CENTRE',
            r'\bCHECK\b': 'CHEQUE',
            r'\bCHECKER\b': 'CHEQUER',
            r'\bCIPHER\b': 'CYPHER',
            r'\bCIVILIZE\b': 'CIVILISE',
            r'\bCOLONIZE\b': 'COLONISE',
            r'\bCOLONIZATION\b': 'COLONISATION',
            r'\bCOLOR\b': 'COLOUR',
            r'\bCOZY\b': 'COSY',
            r'\bCOUNSELOR\b': 'COUNSELLOR',
            r'\bCOUNSELING\b': 'COUNSELLING',
            r'\bCONNECTION\b': 'CONNEXION',
            r'\bDEFENSE\b': 'DEFENCE',
            r'\bDEMAGOG\b': 'DEMAGOGUE',
            r'\bDIALED\b': 'DIALLED',
            r'\bDIALER\b': 'DIALLER',
            r'\bDIALOG\b': 'DIALOGUE',
            r'\bDIARRHEA\b': 'DIARRHOEA',
            r'\bDISK\b': 'DISC',
            r'\bDISTILL\b': 'DISTIL',
            r'\bDONUT\b': 'DOUGHNUT',
            r'\bDRAFT\b': 'DRAUGHT',
            r'\bDREAMED\b': 'DREAMT',
            r'\bEMPHASIZE\b': 'EMPHASISE',
            r'\bENCYCLOPEDIA\b': 'ENCYCLOPAEDIA',
            r'\bENROLLMENT\b': 'ENROLMENT',
            r'\bEQUALING\b': 'EQUALLING',
            r'\bENDEAVOR\b': 'ENDEAVOUR',
            r'\bINQUIRE\b': 'ENQUIRE',
            r'\bFAVORITE\b': 'FAVOURITE',
            r'\bFECES\b': 'FAECES',
            r'\bFIBER\b': 'FIBRE',
            r'\bFETID\b': 'FOETID',
            r'\bFETUS\b': 'FOETUS',
            r'\bFLUTIST\b': 'FLAUTIST',
            r'\bFLAVOR\b': 'FLAVOUR',
            r'\bFULFILL\b': 'FULFIL',
            r'\bFUROR\b': 'FURORE',
            r'\bFUELING\b': 'FUELLING',
            r'\bJAIL\b': 'GAOL',
            r'\bGLYCERIN\b': 'GLYCERINE',
            r'\bGRAY\b': 'GREY',
            r'\bGYNECOLOGY\b': 'GYNAECOLOGY',
            r'\bGENERALIZE\b': 'GENERALISE',
            r'\bHAEMOPHILIA\b': 'HAEMOPHILIA',
            r'\bHEMATOLOGY\b': 'HAEMATOLOGY',
            r'\bHEME\b': 'HAEM',
            r'\bHARBOR\b': 'HARBOUR',
            r'\bHARMONIZE\b': 'HARMONISE',
            r'\bHARMONIZATION\b': 'HARMONISATION',
            r'\bHOMOLOG\b': 'HOMOLOGUE',
            r'\bHONOR\b': 'HONOUR',
            r'\bHUMOR\b': 'HUMOUR',
            r'\bINSTALLMENT\b': 'INSTALMENT',
            r'\bITALICIZE\b': 'ITALICISE',
            r'\bJEWELRY\b': 'JEWELLERY',
            r'\bJUDGMENT\b': 'JUDGEMENT',
            r'\bCURB\b': 'KERB',
            r'\bKILOMETER\b': 'KILOMETRE',
            r'\bLABOR\b': 'LABOUR',
            r'\bLEAPED\b': 'LEAPT',
            r'\bLEARNED\b': 'LEARNT',
            r'\bLEUKEMIA\b': 'LEUKAEMIA',
            r'\bLICENSE\b': 'LICENCE',
            r'\bLICORICE\b': 'LIQUORICE',
            r'\bLITE\b': 'LIGHT',
            r'\bLITER\b': 'LITRE',
            r'\bLODGEMENT\b': 'LODGMENT',
            r'\bLUV\b': 'LOVE',
            r'\bLUVS\b': 'LOVES',
            r'\bMANEUVER\b': 'MANOEUVRE',
            r'\bMARVELOUS\b': 'MARVELLOUS',
            r'\bMETER\b': 'METRE',
            r'\bMODELING\b': 'MODELLING',
            r'\bMOLD\b': 'MOULD',
            r'\bMOLLUSK\b': 'MOLLUSC',
            r'\bMOLT\b': 'MOULT',
            r'\bMOM\b': 'MUM',
            r'\bMONOLOG\b': 'MONOLOGUE',
            r'\bMUSTACHE\b': 'MOUSTACHE',
            r'\bMOISTURIZER\b': 'MOISTURISER',
            r'\bN\b': '&',
            r'\bNEIGHBOR\b': 'NEIGHBOUR',
            r'\bNITE\b': 'NIGHT',
            r'\bNUMBER\b': 'NO',
            r'\bNOONE\b': 'NO ONE',
            r'\bNO-ONE\b': 'NO ONE',
            r'\bOH\b': 'O',
            r'\bENOLOGY\b': 'OENOLOGY',
            r'\bESOPHAGUS\b': 'OESOPHAGUS',
            r'\bESTROGEN\b': 'OESTROGEN',
            r'\bODOR\b': 'ODOUR',
            r'\bOFFENSE\b': 'OFFENCE',
            r'\bOMELET\b': 'OMELETTE',
            r'\bORGANIZATION\b': 'ORGANISATION',
            r'\bORTHOLOG\b': 'ORTHOLOGUE',
            r'\bORTHOPEDIC\b': 'ORTHOPAEDIC',
            r'\bPEDIATRIC\b': 'PAEDIATRIC',
            r'\bPEDOPHILE\b': 'PAEDOPHILE',
            r'\bPAJAMAS\b': 'PYJAMAS',
            r'\bPARALYZE\b': 'PARALYSE',
            r'\bPARLOR\b': 'PARLOUR',
            r'\bPEDAGOG\b': 'PEDAGOGUE',
            r'\bPLOW\b': 'PLOUGH',
            r'\bPRACTICE\b': 'PRACTISE',
            r'\bPRETENSE\b': 'PRETENCE',
            r'\bPRIZE\b': 'PRISE',
            r'\bPROGRAM\b': 'PROGRAMME',
            r'\bPART\b': 'PT',
            r'\bPARTS\b': 'PT',
            r'\bPTS\b': 'PT',
            r'\bPRT\b': 'PT',
            r'\bPRTS\b': 'PT',
            r'\bQUARRELED\b': 'QUARRELLED',
            r'\bQUARRELING\b': 'QUARELLING',
            r'\bREALIZE\b': 'REALISE',
            r'\bREALIZATION\b': 'REALISATION',
            r'\bRIGOR\b': 'RIGOUR',
            r'\bROUTING\b': 'ROUTEING',
            r'\bSAINT\b': 'ST',
            r'\bSAVIOR\b': 'SAVIOUR',
            r'\bSAVORY\b': 'SAVOURY',
            r'\bSKEPTIC\b': 'SCEPTIC',
            r'\bSIGNALING\b': 'SIGNALLING',
            r'\bSKILLFUL\b': 'SKILFUL',
            r'\bSOME ONE\b': 'SOMEONE',
            r'\bSPECIALTY\b': 'SPECIALITY',
            r'\bSPELLED\b': 'SPELT',
            r'\bSPOILED\b': 'SPOILT',
            r'\bSTORY\b': 'STOREY',
            r'\bSULFUR\b': 'SULPHUR',
            r'\bTHEATER\b': 'THEATRE',
            r'\bTHRU\b': 'THROUGH',
            r'\bTIRE\b': 'TYRE',
            r'\bTONITE\b': 'TONIGHT',
            r'\bTRANQUILITY\b': 'TRANQUILLITY',
            r'\bTRAVELED\b': 'TRAVELLED',
            r'\bTRAVELER\b': 'TRAVELLER',
            r'\bTRAVELING\b': 'TRAVELLING',
            r'\bTUMOR\b': 'TUMOUR',
            r'\bURBANIZATION\b': 'URBANISATION',
            r'\bU\b': 'YOU',
            r'\bVALOR\b': 'VALOUR',
            r'\bVENDOR\b': 'VENDER',
            r'\bVISE\b': 'VICE',
            r'\bVITTLE\b': 'VICTUAL',
            r'\bVIGOR\b': 'VIGOUR',
            r'\bVOLUME\b': 'VOL',
            r'\bWINTRY\b': 'WINTERY',
            r'\bWHISKEY\b': 'WHISKY',
            r'\bWOOLEN\b': 'WOOLLEN',
            r'\bYOGURT\b': 'YOGHURT'
        }
        
        # Apply all word translations
        for original, replacement in word_translations.items():
            text = re.sub(original, replacement, text)
            
        return text
    except:
        # If any error occurs, return empty string
        return ""


def balance_brackets(text):
    """
    Balances brackets in a string by adding missing closing brackets or removing extra closing brackets.
    Handles three types of brackets: (), [], {}.
    
    Args:
        text: String that may contain unbalanced brackets
        
    Returns:
        String with balanced brackets
    """
    if not text:
        return ""
    
    # Define bracket pairs
    bracket_pairs = {
        '(': ')',
        '[': ']',
        '{': '}'
    }
    
    # Set of all opening and closing brackets
    opening_brackets = set(bracket_pairs.keys())
    closing_brackets = set(bracket_pairs.values())
    
    # Stack to track open brackets
    stack = []
    
    # Create a list from the string to make character-by-character modifications
    result = list(text)
    
    # First pass: handle missing closing brackets and extra closing brackets
    i = 0
    while i < len(result):
        char = result[i]
        
        if char in opening_brackets:
            # Found an opening bracket, push to stack
            stack.append((char, i))
        elif char in closing_brackets:
            # Found a closing bracket
            if not stack:
                # Extra closing bracket with no matching opening bracket
                # Remove it
                result[i] = ''
                i += 1
                continue
            else:
                # Check if the closing bracket matches the last opening bracket
                last_open, _ = stack[-1]
                expected_close = bracket_pairs[last_open]
                
                if char == expected_close:
                    # Correct closing bracket, pop from stack
                    stack.pop()
                else:
                    # Wrong type of closing bracket
                    # Replace with the correct closing bracket
                    result[i] = expected_close
                    stack.pop()
        
        i += 1
    
    # After processing, add any missing closing brackets
    while stack:
        open_bracket, open_pos = stack.pop()
        # Add the corresponding closing bracket at the end of the string
        result.append(bracket_pairs[open_bracket])
    
    # Convert result list back to string
    return ''.join(result)


def split_artist_name(name):
    
    if not name:
        return []
    
    # Safely convert to string and uppercase
    try:
        original_name = str(name).upper() if name is not None else ""
        # Remove apostrophes and question marks right away
        name_normalized = original_name.replace("'", "").replace("?", "")
        
        # Balance brackets in the normalized name
        name_normalized = balance_brackets(name_normalized)
    except:
        # If any conversion error, return empty list
        return []
    
    all_parts = []
    
    # First, extract content in brackets
    bracket_contents = []
    # Handle different types of brackets
    bracket_pairs = [
        ('(', ')'),  # parentheses
        ('[', ']'),  # square brackets
        ('{', '}')   # curly brackets
    ]
    
    # Make a copy of the name for bracket processing
    working_name = name_normalized
    
    # Define word delimiters for use throughout the function
    word_delimiters = [
        ' FT ', ' FT. ', ' FEAT ', ' FEAT. ', ' WITH ',' AND ', 
        ' VS ', ' VS. ', ' FEATURING ', ' EN ',
        ' AKA ', ' A.K.A. ', ' AKA. '
    ]
    
    # First extract all bracket contents
    for open_bracket, close_bracket in bracket_pairs:
        # Find all instances of content in this bracket type
        start_pos = working_name.find(open_bracket)
        while start_pos != -1:
            # Find matching closing bracket
            end_pos = working_name.find(close_bracket, start_pos + 1)
            if end_pos != -1:  # Only extract if there's a matching closing bracket
                # Extract the content without the brackets
                bracket_content = working_name[start_pos + 1:end_pos].strip()
                if bracket_content:
                    bracket_contents.append(bracket_content)
                
                # Replace the bracketed content without adding spaces - use empty string
                working_name = working_name[:start_pos] + working_name[end_pos + 1:]
            else:
                # No matching closing bracket, move to next possible opening bracket
                start_pos = working_name.find(open_bracket, start_pos + 1)
                continue
                
            # Look for next bracketed content
            start_pos = working_name.find(open_bracket)
    
    # Normalize the main part (without brackets)
    main_part = working_name.strip()
    main_parts = []  # Initialize main_parts as an empty list
    
    # Flag to determine if we found explicit delimiters in the name
    has_explicit_delimiters = False
    
    if main_part:
        # Process main part with delimiters
        # Define character delimiters
        delimiters = ['|', '/', '#', '\\', ',', ';', '"', '`', '-', '_', '&', '+']
        
        # Add spaces for word boundary detection
        main_with_boundaries = ' ' + main_part + ' '
        
        # Check if any delimiters are present in the name
        for d in delimiters:
            if d in main_with_boundaries:
                has_explicit_delimiters = True
                break
        
        if not has_explicit_delimiters:
            for wd in word_delimiters:
                if wd in ' ' + main_with_boundaries + ' ':
                    has_explicit_delimiters = True
                    break
        
        # Replace all delimiters with a standard one
        for d in delimiters:
            main_with_boundaries = main_with_boundaries.replace(d, '|')
        
        # Replace word delimiters
        for wd in word_delimiters:
            if wd in main_with_boundaries:
                main_with_boundaries = main_with_boundaries.replace(wd, '|')
        
        # Split and add to parts
        main_parts = [part.strip() for part in main_with_boundaries.split('|') if part.strip()]
        for part in main_parts:
            clean_part = normalize_special_chars(part)
            if clean_part and len(clean_part) > 1:
                all_parts.append(clean_part)
    
    # Process each bracket content with improved handling for "WITH" and other delimiters
    for content in bracket_contents:
        # First check if content starts with a delimiter word
        delimiter_word_at_start = None
        for delimiter_word in ['WITH', 'FEAT', 'FEAT.', 'FT', 'FT.', 'AND', 'FEATURING']:
            # Need to check with word boundaries (space after it)
            if content.startswith(delimiter_word + ' '):
                delimiter_word_at_start = delimiter_word
                break
        
        if delimiter_word_at_start:
            # Content starts with a delimiter - remove it and process the rest
            rest_of_content = content[len(delimiter_word_at_start):].strip()
            
            # Check for additional delimiters in the rest of content
            has_more_delimiters = False
            
            # Check for character delimiters
            for d in ['|', '/', ',', '&', '+', '-']:
                if d in rest_of_content:
                    has_more_delimiters = True
                    break
            
            # Check for word delimiters
            if not has_more_delimiters:
                for wd in [' AND ', ' WITH ', ' FT ', ' FT. ', ' FEAT ', ' FEAT. ']:
                    if wd in f" {rest_of_content} ":
                        has_more_delimiters = True
                        break
            
            if has_more_delimiters:
                # Replace delimiters with standard one
                cleaned = rest_of_content
                for d in ['|', '/', ',', '&', '+', '-']:
                    cleaned = cleaned.replace(d, '|')
                
                # Replace word delimiters
                for wd in [' AND ', ' WITH ', ' FT ', ' FT. ', ' FEAT ', ' FEAT. ']:
                    if wd in f" {cleaned} ":
                        cleaned = cleaned.replace(wd.strip(), '|')
                
                # Split and add parts
                for part in cleaned.split('|'):
                    part = part.strip()
                    if part:
                        clean_part = normalize_special_chars(part)
                        if clean_part and len(clean_part) > 1:
                            all_parts.append(clean_part)
            else:
                # No more delimiters - add the whole rest of content
                clean_part = normalize_special_chars(rest_of_content)
                if clean_part and len(clean_part) > 1:
                    all_parts.append(clean_part)
        else:
            # No delimiter at start, check if content contains delimiters
            has_delimiter = False
            
            # Check character delimiters
            for d in ['|', '/', ',', '&', '+', '-']:
                if d in content:
                    has_delimiter = True
                    break
            
            # Check word delimiters
            if not has_delimiter:
                for wd in [' AND ', ' WITH ', ' FT ', ' FT. ', ' FEAT ', ' FEAT. ']:
                    if wd in f" {content} ":
                        has_delimiter = True
                        break
            
            if has_delimiter:
                # Content has delimiters - process with standard delimiter handling
                cleaned = content
                for d in ['|', '/', ',', '&', '+', '-']:
                    cleaned = cleaned.replace(d, '|')
                
                # Replace word delimiters
                for wd in [' AND ', ' WITH ', ' FT ', ' FT. ', ' FEAT ', ' FEAT. ']:
                    if wd in f" {cleaned} ":
                        cleaned = cleaned.replace(wd.strip(), '|')
                
                # Split and add parts
                for part in cleaned.split('|'):
                    part = part.strip()
                    if part:
                        clean_part = normalize_special_chars(part)
                        if clean_part and len(clean_part) > 1:
                            all_parts.append(clean_part)
            else:
                # No delimiters - add the whole content
                clean_part = normalize_special_chars(content)
                if clean_part and len(clean_part) > 1:
                    all_parts.append(clean_part)
    
    # Check for name reversals (ONLY if we have exactly 2 main parts and no explicit delimiters)
    if len(main_parts) == 2 and not has_explicit_delimiters:
        reversed_name = f"{main_parts[1]} {main_parts[0]}"
        clean_reversed = normalize_special_chars(reversed_name)
        if clean_reversed and clean_reversed != normalize_special_chars(main_part):
            all_parts.append(clean_reversed)
    
    # Special case for single names: ensure we include the full name as a part
    if len(all_parts) == 0 and main_part:
        clean_full_name = normalize_special_chars(main_part)
        if clean_full_name and len(clean_full_name) > 1:
            all_parts.append(clean_full_name)
    
    # If this is a single name (e.g., "MICHAEL JACKSON"), make sure it's included in the list
    # Only if it's not already included as a result of the delimiter splitting
    if len(main_parts) == 1 and main_part:
        clean_full_name = normalize_special_chars(main_part)
        if clean_full_name and len(clean_full_name) > 1:
            if clean_full_name not in all_parts:
                all_parts.append(clean_full_name)
    
    # Remove duplicates and ensure all parts are clean
    unique_parts = []
    seen = set()
    for part in all_parts:
        # Final cleaning to ensure consistency
        clean_part = part.replace('(', '').replace(')', '').replace('?', '')
        if clean_part and clean_part not in seen and len(clean_part) > 1:
            seen.add(clean_part)
            unique_parts.append(clean_part)
    
    return unique_parts
    

def preprocess_artist_dataframe(df, name_col, id_col, batch_size=1000):
    
    # Convert to pandas for preprocessing
    df_pandas = df.to_pandas()
    total_rows = len(df_pandas)
    
    result_dfs = []
    
    # Process in batches
    for i in range(0, total_rows, batch_size):
        end_idx = min(i + batch_size, total_rows)
        batch = df_pandas.iloc[i:end_idx].copy()
        
        # IMPORTANT: We don't modify the original name column here
        # We keep apostrophes in the original NAME column
        
        # Fill any None values with empty strings to prevent errors
        batch[name_col] = batch[name_col].fillna("")
        
        # Clean artist names - functions will remove apostrophes internally
        batch['CLEAN_NAME'] = batch[name_col].apply(clean_artist_name)
        batch['NORMALIZED_NAME'] = batch[name_col].apply(normalize_special_chars)
        
        # Split artist names and store delimited parts as a list
        batch['DELIMITED_PARTS'] = batch[name_col].apply(lambda x: split_artist_name(x))
        
        result_dfs.append(batch)
            
    # Combine all batches
    combined_df = pd.concat(result_dfs, ignore_index=True)
    
    # Convert back to Snowpark DataFrame
    result_df = df.session.create_dataframe(combined_df)
    
    return result_df


def create_expanded_dataframe(df, name_col, id_col, batch_size=1000):
   
    # Convert to pandas for processing
    df_pandas = df.to_pandas()
    total_rows = len(df_pandas)
    
    expanded_rows = []
    
    # Process in batches
    for i in range(0, total_rows, batch_size):
        end_idx = min(i + batch_size, total_rows)
        batch = df_pandas.iloc[i:end_idx]
        
        batch_expanded_rows = []
        for _, row in batch.iterrows():
            # Add the original name row with apostrophes preserved
            original_row = row.to_dict()
            original_row['IS_VARIANT'] = 'ORIGINAL'
            batch_expanded_rows.append(original_row)
            
            # Skip if there are no delimited parts or DELIMITED_PARTS column doesn't exist
            if 'DELIMITED_PARTS' not in row or not row['DELIMITED_PARTS']:
                continue
            
            # Safely convert the original name to uppercase for comparison
            original_name_upper = ""
            if row[name_col] is not None:
                original_name_upper = str(row[name_col]).upper().replace("'", "")
            
            # Add a row for each delimited part
            for part in row['DELIMITED_PARTS']:
                # Only add if the part is different from the original name
                # and is substantial (more than 2 characters)
                if part and part != original_name_upper and len(part) > 2:
                    variant_row = row.to_dict()
                    variant_row[name_col] = part
                    variant_row['CLEAN_NAME'] = clean_artist_name(part)
                    variant_row['NORMALIZED_NAME'] = normalize_special_chars(part)
                    variant_row['IS_VARIANT'] = 'VARIANT'
                    # Don't include DELIMITED_PARTS in variant rows
                    variant_row['DELIMITED_PARTS'] = []
                    batch_expanded_rows.append(variant_row)
                    
        expanded_rows.extend(batch_expanded_rows)
    
    # Create DataFrame from expanded rows
    expanded_df = pd.DataFrame(expanded_rows)
    
    # Convert back to Snowpark DataFrame
    result_df = df.session.create_dataframe(expanded_df)
    
    return result_df


def filter_data_by_work_ids(session, table_name, work_ids):
    
    # Create a formatted string of work IDs for the SQL query
    work_ids_str = ", ".join([f"'{work_id}'" for work_id in work_ids])
    
    # Query with filter
    query = f"""
    SELECT * 
    FROM {table_name}
    WHERE APRA_WORK_ID IN ({work_ids_str})
    """
    
    # Execute query and return results
    return session.sql(query)


def main(session, mode="example", work_ids=None):
    
    print(f"Running in {mode} mode with improved artist name splitting")
    
    if mode == "example":
        # Create example data
        print("Creating example data...")
        adc_examples = pd.DataFrame({
            'ADC_ARTIST_ID': ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7'],
            'NAME': [
                'AVALON (X-TIAN- DISCOGS (4))',
                'JAMES TAYLOR|ALISON KRAUSS',
                'QUINCY JONES & THE BAND',
                "BJORNSTAD'S KETIL",
                'DE JA VU (ITUNES COVER GROUP)',
                'DOLLY PARTON (WITH BILLY RAY CYRUS AND TANYA TUCK',
                'MIKE LARRY (GOLD)'
            ]
        })
        
        mazooka_examples = pd.DataFrame({
            'RECORDINGS_ID': ['R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7'],
            'ARTIST_NAME': [
                'BOSTON POPS ORCHESTRA',
                'ALISON KRAUSS / JAMES TAYLOR',
                'QUINCY JONES',
                'David Darling / Ketil Bjørnstad',
                'ROY ELDRIDGE AND COUNT BASIE',
                'JUSTIN TIMBERLAKE (FEAT CHRIS STAPLETON AND PHARRELL)',
                '(JOHN LENNON) WITH PAUL MCCARTNEY'
            ]
        })
        
        # Convert to Snowpark DataFrames
        adc_df = session.create_dataframe(adc_examples)
        mazooka_df = session.create_dataframe(mazooka_examples)
        
        # Preprocess ADC data
        print("\nPreprocessing ADC artist data...")
        adc_preprocessed = preprocess_artist_dataframe(adc_df, 'NAME', 'ADC_ARTIST_ID', batch_size=2)
        
        # Preprocess Mazooka data
        print("\nPreprocessing Mazooka artist data...")
        mazooka_preprocessed = preprocess_artist_dataframe(mazooka_df, 'ARTIST_NAME', 'RECORDINGS_ID', batch_size=2)
        
        # Create expanded ADC dataframe
        print("\nCreating expanded ADC dataframe...")
        adc_expanded = create_expanded_dataframe(adc_preprocessed, 'NAME', 'ADC_ARTIST_ID', batch_size=2)
        
        # Create expanded Mazooka dataframe
        print("\nCreating expanded Mazooka dataframe...")
        mazooka_expanded = create_expanded_dataframe(mazooka_preprocessed, 'ARTIST_NAME', 'RECORDINGS_ID', batch_size=2)
        
        # Display results
        print("\nPreprocessed ADC Data:")
        adc_preprocessed_pandas = adc_preprocessed.to_pandas()
        for _, row in adc_preprocessed_pandas.iterrows():
            print(f"ID: {row['ADC_ARTIST_ID']}, Name: {row['NAME']}")
            print(f"  Clean Name: {row['CLEAN_NAME']}")
            print(f"  Normalized Name: {row['NORMALIZED_NAME']}")
            print(f"  Delimited Parts: {row['DELIMITED_PARTS']}")
            print()
            
        print("\nExpanded ADC Data:")
        adc_expanded_pandas = adc_expanded.to_pandas()
        for _, row in adc_expanded_pandas.iterrows():
            print(f"ID: {row['ADC_ARTIST_ID']}, Name: {row['NAME']}, Variant: {row['IS_VARIANT']}")
            print(f"  Clean Name: {row['CLEAN_NAME']}")
            print()
        
    elif mode == "subset":
        # Run with a subset of data
        sample_percentage = 5
        print(f"Running with {sample_percentage}% of Snowflake data")
        
        # Define table names
        adc_table = "EDW_APPS.MATCHING.ADC_ARTISTS_MATCHED_ISWC_VW"
        mazooka_table = "EDW_APPS.MATCHING.MAZOOKA_RECORDINGS_MATCHED_ISWC_VW"
        
        # Intermediate tables
        adc_preprocessed_table = "EDW_APPS.MATCHING.ADC_ARTISTS_PREPROCESSED_TEMP"
        mazooka_preprocessed_table = "EDW_APPS.MATCHING.MAZOOKA_ARTISTS_PREPROCESSED_TEMP"
        
        # Final tables
        adc_expanded_table = "EDW_APPS.MATCHING.ADC_ARTISTS_EXPANDED"
        mazooka_expanded_table = "EDW_APPS.MATCHING.MAZOOKA_ARTISTS_EXPANDED"
        
        # Read from Snowflake tables
        print("Reading data from Snowflake tables...")
        adc_df = session.table(adc_table)
        mazooka_df = session.table(mazooka_table)
        
        # Apply sampling
        adc_count = adc_df.count()
        adc_sample_size = int(adc_count * sample_percentage / 100)
        adc_df = adc_df.sample(n=min(adc_sample_size, adc_count))
        
        mazooka_count = mazooka_df.count()
        mazooka_sample_size = int(mazooka_count * sample_percentage / 100)
        mazooka_df = mazooka_df.sample(n=min(mazooka_sample_size, mazooka_count))
        
        print(f"Data size: {adc_df.count()} ADC artists and {mazooka_df.count()} Mazooka artists")
        
        # Step 1: Preprocess ADC data
        print("\nPreprocessing ADC artist data...")
        adc_preprocessed = preprocess_artist_dataframe(adc_df, 'NAME', 'ADC_ARTIST_ID')
        
        # Save intermediate results
        print(f"Saving preprocessed ADC data to {adc_preprocessed_table}...")
        adc_preprocessed.write.mode("overwrite").save_as_table(adc_preprocessed_table)
        
        # Step 2: Preprocess Mazooka data
        print("\nPreprocessing Mazooka artist data...")
        mazooka_preprocessed = preprocess_artist_dataframe(mazooka_df, 'ARTIST_NAME', 'RECORDINGS_ID')
        
        # Save intermediate results
        print(f"Saving preprocessed Mazooka data to {mazooka_preprocessed_table}...")
        mazooka_preprocessed.write.mode("overwrite").save_as_table(mazooka_preprocessed_table)
        
        # Step 3: Create expanded ADC dataframe
        print("\nCreating expanded ADC dataframe...")
        adc_preprocessed = session.table(adc_preprocessed_table)
        adc_expanded = create_expanded_dataframe(adc_preprocessed, 'NAME', 'ADC_ARTIST_ID')
        
        # Save final ADC results
        print(f"Saving expanded ADC data to {adc_expanded_table}...")
        adc_expanded.write.mode("overwrite").save_as_table(adc_expanded_table)
        
        # Step 4: Create expanded Mazooka dataframe
        print("\nCreating expanded Mazooka dataframe...")
        mazooka_preprocessed = session.table(mazooka_preprocessed_table)
        mazooka_expanded = create_expanded_dataframe(mazooka_preprocessed, 'ARTIST_NAME', 'RECORDINGS_ID')
        
        # Save final Mazooka results
        print(f"Saving expanded Mazooka data to {mazooka_expanded_table}...")
        mazooka_expanded.write.mode("overwrite").save_as_table(mazooka_expanded_table)
        
        # Clean up intermediate tables
        print("\nCleaning up intermediate tables...")
        session.sql(f"DROP TABLE IF EXISTS {adc_preprocessed_table}").collect()
        session.sql(f"DROP TABLE IF EXISTS {mazooka_preprocessed_table}").collect()
        
        print("Preprocessing completed successfully!")
        
    elif mode == "full":
        # Run with full dataset
        print("Running with FULL Snowflake dataset...")
        
        # Define table names
        adc_table = "EDW_APPS.MATCHING.ADC_ARTISTS_MATCHED_ISWC_VW"
        mazooka_table = "EDW_APPS.MATCHING.MAZOOKA_RECORDINGS_MATCHED_ISWC_VW"
        
        # Intermediate tables
        adc_preprocessed_table = "EDW_APPS.MATCHING.ADC_ARTISTS_PREPROCESSED_TEMP"
        mazooka_preprocessed_table = "EDW_APPS.MATCHING.MAZOOKA_ARTISTS_PREPROCESSED_TEMP"
        
        # Final tables
        adc_expanded_table = "EDW_APPS.MATCHING.ADC_ARTISTS_EXPANDED"
        mazooka_expanded_table = "EDW_APPS.MATCHING.MAZOOKA_ARTISTS_EXPANDED"
        
        # Read from Snowflake tables
        print("Reading data from Snowflake tables...")
        adc_df = session.table(adc_table)
        mazooka_df = session.table(mazooka_table)
        
        print(f"Data size: {adc_df.count()} ADC artists and {mazooka_df.count()} Mazooka artists")
        
        # Step 1: Preprocess ADC data
        print("\nPreprocessing ADC artist data...")
        adc_preprocessed = preprocess_artist_dataframe(adc_df, 'NAME', 'ADC_ARTIST_ID', batch_size=5000)
        
        # Save intermediate results
        print(f"Saving preprocessed ADC data to {adc_preprocessed_table}...")
        adc_preprocessed.write.mode("overwrite").save_as_table(adc_preprocessed_table)
        
        # Step 2: Preprocess Mazooka data
        print("\nPreprocessing Mazooka artist data...")
        mazooka_preprocessed = preprocess_artist_dataframe(mazooka_df, 'ARTIST_NAME', 'RECORDINGS_ID', batch_size=5000)
        
        # Save intermediate results
        print(f"Saving preprocessed Mazooka data to {mazooka_preprocessed_table}...")
        mazooka_preprocessed.write.mode("overwrite").save_as_table(mazooka_preprocessed_table)
        
        # Step 3: Create expanded ADC dataframe
        print("\nCreating expanded ADC dataframe...")
        adc_preprocessed = session.table(adc_preprocessed_table)
        adc_expanded = create_expanded_dataframe(adc_preprocessed, 'NAME', 'ADC_ARTIST_ID', batch_size=5000)
        
        # Save final ADC results
        print(f"Saving expanded ADC data to {adc_expanded_table}...")
        adc_expanded.write.mode("overwrite").save_as_table(adc_expanded_table)
        
        # Step 4: Create expanded Mazooka dataframe
        print("\nCreating expanded Mazooka dataframe...")
        mazooka_preprocessed = session.table(mazooka_preprocessed_table)
        mazooka_expanded = create_expanded_dataframe(mazooka_preprocessed, 'ARTIST_NAME', 'RECORDINGS_ID', batch_size=5000)
        
        # Save final Mazooka results
        print(f"Saving expanded Mazooka data to {mazooka_expanded_table}...")
        mazooka_expanded.write.mode("overwrite").save_as_table(mazooka_expanded_table)
        
        # Clean up intermediate tables
        print("\nCleaning up intermediate tables...")
        session.sql(f"DROP TABLE IF EXISTS {adc_preprocessed_table}").collect()
        session.sql(f"DROP TABLE IF EXISTS {mazooka_preprocessed_table}").collect()
        
        print("Preprocessing completed successfully!")
    
    elif mode == "work":
        # Run for specific APRA work IDs
        if not work_ids or len(work_ids) == 0:
            print("No work IDs provided. Please provide a list of APRA_WORK_ID values.")
            return
            
        print(f"Running for {len(work_ids)} specific APRA work IDs: {work_ids}")
        
        # Define table names
        adc_table = "EDW_APPS.MATCHING.ADC_ARTISTS_MATCHED_ISWC_VW"
        mazooka_table = "EDW_APPS.MATCHING.MAZOOKA_RECORDINGS_MATCHED_ISWC_VW"
        
        # Intermediate tables
        adc_preprocessed_table = "EDW_APPS.MATCHING.ADC_ARTISTS_PREPROCESSED_WORK_TEMP"
        mazooka_preprocessed_table = "EDW_APPS.MATCHING.MAZOOKA_ARTISTS_PREPROCESSED_WORK_TEMP"
        
        # Final tables
        adc_expanded_table = "EDW_APPS.MATCHING.ADC_ARTISTS_EXPANDED"
        mazooka_expanded_table = "EDW_APPS.MATCHING.MAZOOKA_ARTISTS_EXPANDED"
        
        # Filter data for the specified work IDs
        print("Filtering data for specified work IDs...")
        adc_df = filter_data_by_work_ids(session, adc_table, work_ids)
        mazooka_df = filter_data_by_work_ids(session, mazooka_table, work_ids)
        
        print(f"Filtered data size: {adc_df.count()} ADC artists and {mazooka_df.count()} Mazooka artists")
        
        if adc_df.count() == 0 or mazooka_df.count() == 0:
            print("Warning: One or both datasets have no records for the specified work IDs.")
            if adc_df.count() == 0:
                print("No records found in ADC data for the specified work IDs.")
            if mazooka_df.count() == 0:
                print("No records found in Mazooka data for the specified work IDs.")
            return
            
        # Step 1: Preprocess ADC data
        print("\nPreprocessing ADC artist data...")
        adc_preprocessed = preprocess_artist_dataframe(adc_df, 'NAME', 'ADC_ARTIST_ID', batch_size=1000)
        
        # Save intermediate results
        print(f"Saving preprocessed ADC data to {adc_preprocessed_table}...")
        adc_preprocessed.write.mode("overwrite").save_as_table(adc_preprocessed_table)
        
        # Step 2: Preprocess Mazooka data
        print("\nPreprocessing Mazooka artist data...")
        mazooka_preprocessed = preprocess_artist_dataframe(mazooka_df, 'ARTIST_NAME', 'RECORDINGS_ID', batch_size=1000)
        
        # Save intermediate results
        print(f"Saving preprocessed Mazooka data to {mazooka_preprocessed_table}...")
        mazooka_preprocessed.write.mode("overwrite").save_as_table(mazooka_preprocessed_table)
        
        # Step 3: Create expanded ADC dataframe
        print("\nCreating expanded ADC dataframe...")
        adc_preprocessed = session.table(adc_preprocessed_table)
        adc_expanded = create_expanded_dataframe(adc_preprocessed, 'NAME', 'ADC_ARTIST_ID', batch_size=1000)
        
        # Save final ADC results
        print(f"Saving expanded ADC data to {adc_expanded_table}...")
        adc_expanded.write.mode("overwrite").save_as_table(adc_expanded_table)
        
        # Step 4: Create expanded Mazooka dataframe
        print("\nCreating expanded Mazooka dataframe...")
        mazooka_preprocessed = session.table(mazooka_preprocessed_table)
        mazooka_expanded = create_expanded_dataframe(mazooka_preprocessed, 'ARTIST_NAME', 'RECORDINGS_ID', batch_size=1000)
        
        # Save final Mazooka results
        print(f"Saving expanded Mazooka data to {mazooka_expanded_table}...")
        mazooka_expanded.write.mode("overwrite").save_as_table(mazooka_expanded_table)
        
        # Clean up intermediate tables
        print("\nCleaning up intermediate tables...")
        session.sql(f"DROP TABLE IF EXISTS {adc_preprocessed_table}").collect()
        session.sql(f"DROP TABLE IF EXISTS {mazooka_preprocessed_table}").collect()
        
        print("Preprocessing for specific work IDs completed successfully!")

        
if __name__ == "__main__":

    work_ids = [
        'GW00577031' #5858
        #'GW01412959', #2240
        #'GW34081705', #1407
        #'GW01602130', #770
        #'GW50027690', #130
        #'GW26412932',
    ]
    
    # Run with specific work IDs
    main(session, "work", work_ids)

    # Run with examples
    # main(session, "example")
    
    # Run with subset of data (5%)
    # main(session, "subset")
    
    # Run with full dataset (caution: could be resource-intensive)
    # main(session, "full")

In [ ]:
#DELIMITER COUNT

import pandas as pd
import numpy as np
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, array_size, split, length, when

def get_delimited_parts_count(session, table_name, delimited_parts_col):
    """
    Get the count of delimited parts by determining the column type and using appropriate method.
    
    Args:
        session: Snowflake session
        table_name: Name of the table containing the delimited_parts column
        delimited_parts_col: Name of the column containing delimited parts
        
    Returns:
        SQL snippet for getting the count
    """
    # Get a sample row to check the format
    sample_query = f"SELECT {delimited_parts_col} FROM {table_name} WHERE {delimited_parts_col} IS NOT NULL LIMIT 1"
    sample_result = session.sql(sample_query).collect()
    
    if not sample_result:
        return f"LENGTH({delimited_parts_col})"  # Fallback
    
    sample_row = sample_result[0][delimited_parts_col]
    
    # Determine if it's an array or string
    if isinstance(sample_row, list):
        # It's an actual array
        return f"ARRAY_SIZE({delimited_parts_col})"
    elif isinstance(sample_row, str) and (sample_row.startswith('[') and sample_row.endswith(']')):
        # It's a string representation of an array, can use PARSE_JSON
        return f"ARRAY_SIZE(PARSE_JSON({delimited_parts_col}))"
    elif isinstance(sample_row, str) and (',' in sample_row):
        # It's a string with comma delimiters
        return f"ARRAY_SIZE(SPLIT({delimited_parts_col}, ','))"
    else:
        # For any other case, use a simple word count as an approximation
        return f"ARRAY_SIZE(SPLIT(TRIM({delimited_parts_col}), ' '))"

def match_adc_artists_by_work_id(session, specific_work_ids=None):
    """
    Match artists from ADC and Mazooka expanded tables based on APRA_WORK_ID.
    Includes DELIMITED_PARTS column in the output for ADC.
    For artist count > 1, sets the count as n-1.
    
    Args:
        session: Snowflake session
        specific_work_ids: Optional list of specific APRA_WORK_ID values to filter by
    """
    # Define table names for the full dataset
    adc_expanded_table = "EDW_APPS.MATCHING.ADC_ARTISTS_EXPANDED"
    mazooka_expanded_table = "EDW_APPS.MATCHING.MAZOOKA_ARTISTS_EXPANDED"
    output_table = "EDW_APPS.MATCHING.ADC_ARTIST_DELIMITER_COUNT"  # Renamed as requested
    
    # Determine how to count delimited parts
    adc_delimited_parts_col = 'DELIMITED_PARTS'
    adc_delimited_parts_count = get_delimited_parts_count(
        session, adc_expanded_table, adc_delimited_parts_col
    )
    
    # Prepare the work_id filter clause
    work_id_filter = ""
    if specific_work_ids and len(specific_work_ids) > 0:
        work_ids_str = ", ".join([f"'{work_id}'" for work_id in specific_work_ids])
        work_id_filter = f"AND APRA_WORK_ID IN ({work_ids_str})"
    
    # Execute a simple join query including the DELIMITED_PARTS column
    # Modify the count to be n-1 if n > 1
    matching_query = f"""
    WITH adc_artists AS (
        SELECT 
            APRA_WORK_ID,
            ADC_ARTIST_ID AS APRA_ARTIST_ID,
            NAME AS APRA_NAME,
            {adc_delimited_parts_count} AS APRA_ARTIST_CT,
            {adc_delimited_parts_col} AS DELIMITED_PARTS
        FROM {adc_expanded_table}
        WHERE APRA_WORK_ID IS NOT NULL
        AND IS_VARIANT = 'ORIGINAL'
        {work_id_filter}
    ),
    mazooka_artists AS (
        SELECT 
            APRA_WORK_ID,
            RECORDINGS_ID AS MUZOOKA_TRACK_ID,
            ARTIST_NAME AS MUZOOKA_NAME
        FROM {mazooka_expanded_table}
        WHERE APRA_WORK_ID IS NOT NULL
        AND IS_VARIANT = 'ORIGINAL'
        {work_id_filter}
    )
    SELECT 
        a.APRA_WORK_ID,
        m.MUZOOKA_TRACK_ID,
        a.APRA_ARTIST_ID,
        a.APRA_NAME,
        a.DELIMITED_PARTS,
        a.APRA_ARTIST_CT
    FROM adc_artists a
    JOIN mazooka_artists m ON a.APRA_WORK_ID = m.APRA_WORK_ID
    """
    
    # Execute the query and save results
    matching_results = session.sql(matching_query)
    matching_results.write.mode("overwrite").save_as_table(output_table)

def match_mazooka_artists_by_work_id(session, specific_work_ids=None):
    """
    Match artists from Mazooka and ADC expanded tables based on APRA_WORK_ID.
    Includes DELIMITED_PARTS column in the output for Mazooka.
    For artist count > 1, sets the count as n-1.
    
    Args:
        session: Snowflake session
        specific_work_ids: Optional list of specific APRA_WORK_ID values to filter by
    """
    # Define table names for the full dataset
    adc_expanded_table = "EDW_APPS.MATCHING.ADC_ARTISTS_EXPANDED"
    mazooka_expanded_table = "EDW_APPS.MATCHING.MAZOOKA_ARTISTS_EXPANDED"
    output_table = "EDW_APPS.MATCHING.MZK_ARTIST_DELIMITER_COUNT"  # New output table for Mazooka
    
    # Determine how to count delimited parts for Mazooka
    mzk_delimited_parts_col = 'DELIMITED_PARTS'
    mzk_delimited_parts_count = get_delimited_parts_count(
        session, mazooka_expanded_table, mzk_delimited_parts_col
    )
    
    # Prepare the work_id filter clause
    work_id_filter = ""
    if specific_work_ids and len(specific_work_ids) > 0:
        work_ids_str = ", ".join([f"'{work_id}'" for work_id in specific_work_ids])
        work_id_filter = f"AND APRA_WORK_ID IN ({work_ids_str})"
    
    # Execute a simple join query including the DELIMITED_PARTS column for Mazooka
    # Modify the count to be n-1 if n > 1
    matching_query = f"""
    WITH adc_artists AS (
        SELECT 
            APRA_WORK_ID,
            ADC_ARTIST_ID AS APRA_ARTIST_ID,
            NAME AS APRA_NAME
        FROM {adc_expanded_table}
        WHERE APRA_WORK_ID IS NOT NULL
        AND IS_VARIANT = 'ORIGINAL'
        {work_id_filter}
    ),
    mazooka_artists AS (
        SELECT 
            APRA_WORK_ID,
            RECORDINGS_ID AS MUZOOKA_TRACK_ID,
            ARTIST_NAME AS MUZOOKA_NAME,
            {mzk_delimited_parts_count} AS MUZOOKA_ARTIST_CT,
            {mzk_delimited_parts_col} AS DELIMITED_PARTS
        FROM {mazooka_expanded_table}
        WHERE APRA_WORK_ID IS NOT NULL
        AND IS_VARIANT = 'ORIGINAL'
        {work_id_filter}
    )
    SELECT 
        m.APRA_WORK_ID,
        m.MUZOOKA_TRACK_ID,
        a.APRA_ARTIST_ID,
        m.MUZOOKA_NAME,
        m.DELIMITED_PARTS,
        m.MUZOOKA_ARTIST_CT
    FROM mazooka_artists m
    JOIN adc_artists a ON m.APRA_WORK_ID = a.APRA_WORK_ID
    """
    
    # Execute the query and save results
    matching_results = session.sql(matching_query)
    matching_results.write.mode("overwrite").save_as_table(output_table)

def main(session):
   
    #Run for all APRA work IDs
    match_adc_artists_by_work_id(session)
    match_mazooka_artists_by_work_id(session)
    
    # Run for specific APRA work IDs (uncomment and modify list as needed)
    # work_ids = [
    #     'GW03136850',  # Replace with actual work IDs
    #     'GW00577031'
    # ]
    # match_adc_artists_by_work_id(session, work_ids)
    # match_mazooka_artists_by_work_id(session, work_ids)

if __name__ == "__main__":
    main(session)

In [ ]:
#FINAL MATCHING


from snowflake.snowpark.functions import sum as sum_func, regexp_replace, upper, trim
from snowflake.snowpark.types import StringType, FloatType, ArrayType, MapType, VariantType


create_udf_sql = """
CREATE OR REPLACE FUNCTION MATCH_ARTIST_NAMES(adc_parts STRING, mzk_parts STRING)
RETURNS FLOAT
LANGUAGE JAVASCRIPT
AS
$$

function matchArtistNames(adc_parts, mzk_parts) {
    try {
        // Parse the JSON strings into arrays
        let adcArtists = JSON.parse(adc_parts);
        let mzkArtists = JSON.parse(mzk_parts);
        
        // Clean the arrays - handle escaped delimiters and empty entries
        adcArtists = cleanArtistArray(adcArtists);
        mzkArtists = cleanArtistArray(mzkArtists);
        
        // Initialize total score sum
        let totalScore = 0;
        
        // For each ADC artist, find the best matching MZK artist
        for (let i = 0; i < adcArtists.length; i++) {
            let adcArtist = adcArtists[i].toString().trim().toUpperCase();
            let bestScore = 0;
            
            // Try matching with each MZK artist
            for (let j = 0; j < mzkArtists.length; j++) {
                let mzkArtist = mzkArtists[j].toString().trim().toUpperCase();
                let score = 0;
                
                // Check for exact match
                if (adcArtist === mzkArtist) {
                    score = 1.0;
                } 
                // Check for reversed name match (e.g., "FIRST LAST" vs "LAST FIRST")
                else if (adcArtist.includes(' ') && mzkArtist.includes(' ')) {
                    let adcParts = adcArtist.split(' ');
                    let mzkParts = mzkArtist.split(' ');
                    
                    // Simple check for reversed two-part names
                    if (adcParts.length === 2 && mzkParts.length === 2 &&
                        adcParts[0] === mzkParts[1] && adcParts[1] === mzkParts[0]) {
                        score = 1.0;
                    }
                    else {
                        // Fuzzy matching
                        score = jaroWinklerSimilarity(adcArtist, mzkArtist);
                    }
                }
                else {
                    // Fuzzy matching
                    score = jaroWinklerSimilarity(adcArtist, mzkArtist);
                }
                
                // Update best score if this match is better
                if (score > bestScore) {
                    bestScore = score;
                }
            }
            
            // Only add to total if the individual match score is > 0.8
            if (bestScore > 0.8) {
                totalScore += bestScore;
            } else {
                // Assign 0 for poor matches
                totalScore += 0;
            }
        }
        
        // Return the sum of all filtered best scores
        return totalScore;
    } catch (e) {
        // Return error details if something goes wrong
        return 0; // Return 0 on error
    }
}

// New function to clean artist arrays - removes invalid entries from backslash escapes
function cleanArtistArray(artists) {
    // Filter out entries that contain escaped delimiters (like "#\\")
    return artists.filter(artist => {
        // If the artist name contains a hash followed by backslashes, it's likely an escaped delimiter
        // and not a real artist name component
        let str = artist.toString().trim();
        
        // Check if this entry is just the original name with an escaped delimiter
        // This pattern matches strings that contain "#\" or similar escaped sequences
        if (/^[^#]+#\\+$/.test(str)) {
            return false;
        }
        
        // Keep entries that are actual artist names
        return str.length > 0;
    });
}

// Jaro-Winkler similarity implementation remains the same
function jaroWinklerSimilarity(s1, s2) {
    // If either string is empty, return 0
    if (!s1 || !s2) {
        return 0;
    }
    
    // If both strings are identical, return 1
    if (s1 === s2) {
        return 1;
    }
    
    // Calculate Jaro Distance
    let m = 0; // matching characters
    let t = 0; // transpositions
    let range = Math.floor(Math.max(s1.length, s2.length) / 2) - 1;
    range = Math.max(0, range); // Ensure range is not negative
    
    let s1Matches = new Array(s1.length).fill(false);
    let s2Matches = new Array(s2.length).fill(false);
    
    // Find matching characters within range
    for (let i = 0; i < s1.length; i++) {
        let start = Math.max(0, i - range);
        let end = Math.min(i + range + 1, s2.length);
        
        for (let j = start; j < end; j++) {
            if (!s2Matches[j] && s1[i] === s2[j]) {
                s1Matches[i] = true;
                s2Matches[j] = true;
                m++;
                break;
            }
        }
    }
    
    // If no matching characters, return 0
    if (m === 0) {
        return 0;
    }
    
    // Count transpositions
    let k = 0;
    for (let i = 0; i < s1.length; i++) {
        if (s1Matches[i]) {
            while (!s2Matches[k]) {
                k++;
            }
            
            if (s1[i] !== s2[k]) {
                t++;
            }
            
            k++;
        }
    }
    
    // Calculate Jaro similarity
    t = Math.floor(t / 2);
    let jaroSim = (m / s1.length + m / s2.length + (m - t) / m) / 3;
    
    // Calculate Jaro-Winkler similarity
    let p = 0.1; // scaling factor
    let l = 0;   // length of common prefix (max 4)
    
    // Count common prefix up to 4 characters
    for (let i = 0; i < Math.min(4, Math.min(s1.length, s2.length)); i++) {
        if (s1[i] === s2[i]) {
            l++;
        } else {
            break;
        }
    }
    
    // Apply Winkler modification
    return jaroSim + (l * p * (1 - jaroSim));
}

return matchArtistNames(ADC_PARTS, MZK_PARTS);
$$;
"""

# Execute the SQL to create the UDF
session.sql(create_udf_sql).collect()


# Load the data
adc_df = session.table("ADC_ARTIST_DELIMITER_COUNT")
mzk_df = session.table("MZK_ARTIST_DELIMITER_COUNT")

# Create temporary views to make SQL operations easier
adc_df.create_or_replace_temp_view("ADC_TEMP_VIEW")
mzk_df.create_or_replace_temp_view("MZK_TEMP_VIEW")


match_sql = """
SELECT DISTINCT
    a.APRA_WORK_ID,
    a.MUZOOKA_TRACK_ID,
    a.APRA_ARTIST_ID,
    a.APRA_NAME,
    UPPER(m.MUZOOKA_NAME) AS MUZOOKA_NAME,
    a.DELIMITED_PARTS as ADC_DELIMITED_PARTS,
    a.APRA_ARTIST_CT,
    m.DELIMITED_PARTS as MZK_DELIMITED_PARTS,
    m.MUZOOKA_ARTIST_CT,
    MATCH_ARTIST_NAMES(a.DELIMITED_PARTS, m.DELIMITED_PARTS) as VARIANT_MATCH_SCORE,
    (VARIANT_MATCH_SCORE/a.APRA_ARTIST_CT) as ARTIST_MATCH_SCORE
FROM 
    ADC_TEMP_VIEW a
INNER JOIN 
    MZK_TEMP_VIEW m
ON 
    a.APRA_WORK_ID = m.APRA_WORK_ID
WHERE
    MATCH_ARTIST_NAMES(a.DELIMITED_PARTS, m.DELIMITED_PARTS) >= 0.8 AND ARTIST_MATCH_SCORE >= 0.8
"""

# Execute the SQL query
matched_df = session.sql(match_sql)

# Create or replace the output table
matched_df.write.mode("overwrite").save_as_table("ARTIST_NAME_MATCH_RESULTS")

# Show the results
matched_df.show()

summary_sql = """
SELECT
    COUNT(CASE WHEN ARTIST_MATCH_SCORE = 1.0 THEN 1 END) AS EXACT_MATCH_COUNT,
    COUNT(CASE WHEN ARTIST_MATCH_SCORE < 1.0 AND ARTIST_MATCH_SCORE >= 0.9 THEN 1 END) AS HIGH_MATCH_COUNT,
    COUNT(CASE WHEN ARTIST_MATCH_SCORE < 0.9 AND ARTIST_MATCH_SCORE >= 0.8 THEN 1 END) AS MEDIUM_MATCH_COUNT,
    COUNT(*) AS TOTAL_MATCHES
FROM
    ARTIST_NAME_MATCH_RESULTS
"""

debug_sql = """
SELECT 
    ROUND(ARTIST_MATCH_SCORE, 2) as SCORE_RANGE,
    COUNT(*) as COUNT
FROM 
    ARTIST_NAME_MATCH_RESULTS
GROUP BY 
    ROUND(ARTIST_MATCH_SCORE, 2)
ORDER BY 
    SCORE_RANGE DESC
"""

debug_df = session.sql(debug_sql)
print("Score distribution:")
debug_df.show()

summary_df = session.sql(summary_sql)
summary_df.show()

print("Processing complete. Results stored in ARTIST_NAME_MATCH_RESULTS table.")
